# Named Entity Recognition using spaCy and Custom Entity Training

**Aim:** To perform Named Entity Recognition (NER) on news text with a pre-trained spaCy model, and to train and
test a custom NER model for a domain-specific task (medical text: diseases, medications, treatments).

**Steps:** load a pre-trained model → extract and display entities → train a custom NER model →
**test** both models with hand-labelled data (precision, recall, F1).

In [ ]:
!pip install -q spacy
!python -m spacy download en_core_web_sm

In [ ]:
import random
import spacy
import pandas as pd
import matplotlib.pyplot as plt

from spacy import displacy
from spacy.training import Example

nlp = spacy.load("en_core_web_sm")
print("spaCy model loaded successfully!")

## 1. Pre-trained model: extract named entities

In [ ]:
text = """
Apple Inc. announced on September 1, 2026 that it will invest $5 billion
in a new technology center in California. CEO Tim Cook said the company
will create more than 10,000 jobs over the next five years.
The new center will be located near San Francisco and will focus on
artificial intelligence and semiconductor research. Microsoft and Google
are also expanding their AI operations in the United States.
"""

doc = nlp(text)
print("Text processed successfully!")
print("Pipeline components:", nlp.pipe_names)

In [ ]:
print("========== NAMED ENTITIES ==========\n")
for ent in doc.ents:
    print(f"Entity: {ent.text:<25} Label: {ent.label_:<10} Description: {spacy.explain(ent.label_)}")

In [ ]:
for label in ["ORG", "GPE", "PERSON", "MONEY"]:
    print(f"{label:<7}:", spacy.explain(label))

In [ ]:
displacy.render(doc, style="ent", jupyter=True)

In [ ]:
people = [ent.text for ent in doc.ents if ent.label_ == "PERSON"]
organizations = [ent.text for ent in doc.ents if ent.label_ == "ORG"]
locations = [ent.text for ent in doc.ents if ent.label_ in ["GPE", "LOC"]]

print("People       :", people)
print("Organizations:", organizations)
print("Locations    :", locations)

In [ ]:
entity_df = pd.DataFrame([
    {"Entity": ent.text, "Label": ent.label_, "Description": spacy.explain(ent.label_)}
    for ent in doc.ents
])
display(entity_df)

# How many entities of each type?
counts = entity_df["Label"].value_counts()
display(counts.to_frame("Count"))

counts.plot(kind="bar", figsize=(8, 4), title="Entities found by type")
plt.ylabel("Count")
plt.show()

# Note: the small model labels 'AI' as GPE (a place). This is a real model mistake and is
# measured in the testing section below.

## 2. Custom NER model: diseases, medications and treatments

In [ ]:
def make_example(sentence, **entities):
    """Build a spaCy training example. Character positions are found automatically,
    so they can never be off by one.   make_example("...", DISEASE=["asthma"])"""
    spans = []
    for label, items in entities.items():
        for item in items:
            start = sentence.find(item)
            assert start >= 0, f"'{item}' not found in: {sentence}"
            spans.append((start, start + len(item), label))
    return (sentence, {"entities": sorted(spans)})

TRAIN_DATA = [
    make_example("The patient was diagnosed with diabetes and prescribed Metformin.", DISEASE=["diabetes"], MEDICATION=["Metformin"]),
    make_example("The doctor prescribed Insulin to treat diabetes.", MEDICATION=["Insulin"], DISEASE=["diabetes"]),
    make_example("Hypertension can be treated using Amlodipine.", DISEASE=["Hypertension"], MEDICATION=["Amlodipine"]),
    make_example("The patient has asthma and uses Salbutamol.", DISEASE=["asthma"], MEDICATION=["Salbutamol"]),
    make_example("Doctors are studying treatments for cancer using chemotherapy.", DISEASE=["cancer"], TREATMENT=["chemotherapy"]),
    make_example("She was given Paracetamol for her fever.", MEDICATION=["Paracetamol"], DISEASE=["fever"]),
    make_example("Aspirin is often used to prevent heart disease.", MEDICATION=["Aspirin"], DISEASE=["heart disease"]),
    make_example("He suffers from migraine and takes Ibuprofen.", DISEASE=["migraine"], MEDICATION=["Ibuprofen"]),
    make_example("Radiotherapy was recommended for the tumor.", TREATMENT=["Radiotherapy"], DISEASE=["tumor"]),
    make_example("The child with pneumonia received Amoxicillin.", DISEASE=["pneumonia"], MEDICATION=["Amoxicillin"]),
    make_example("Physiotherapy helps patients recover from arthritis.", TREATMENT=["Physiotherapy"], DISEASE=["arthritis"]),
    make_example("Atorvastatin lowers cholesterol in patients with high risk.", MEDICATION=["Atorvastatin"], DISEASE=["cholesterol"]),
    make_example("Surgery is the main treatment for appendicitis.", TREATMENT=["Surgery"], DISEASE=["appendicitis"]),
    make_example("The doctor treated his depression with Sertraline.", DISEASE=["depression"], MEDICATION=["Sertraline"]),
    make_example("Dialysis is required for patients with kidney failure.", TREATMENT=["Dialysis"], DISEASE=["kidney failure"]),
    make_example("Patients with epilepsy are given Carbamazepine.", DISEASE=["epilepsy"], MEDICATION=["Carbamazepine"]),
    make_example("Immunotherapy is a new treatment for melanoma.", TREATMENT=["Immunotherapy"], DISEASE=["melanoma"]),
    make_example("She takes Levothyroxine because of hypothyroidism.", MEDICATION=["Levothyroxine"], DISEASE=["hypothyroidism"]),
    make_example("Antibiotics such as Azithromycin treat bronchitis.", MEDICATION=["Azithromycin"], DISEASE=["bronchitis"]),
    make_example("The patient with malaria was treated with Chloroquine.", DISEASE=["malaria"], MEDICATION=["Chloroquine"]),
    make_example("Vaccination can prevent measles.", TREATMENT=["Vaccination"], DISEASE=["measles"]),
    make_example("Doctors used laser therapy to treat his glaucoma.", TREATMENT=["laser therapy"], DISEASE=["glaucoma"]),
    make_example("Warfarin is prescribed to prevent stroke.", MEDICATION=["Warfarin"], DISEASE=["stroke"]),
    make_example("Counselling is helpful for anxiety.", TREATMENT=["Counselling"], DISEASE=["anxiety"]),
]

# Held-out sentences used ONLY for testing (never shown to the model while training)
TEST_DATA = [
    make_example("The patient has diabetes and hypertension.", DISEASE=["diabetes", "hypertension"]),
    make_example("The doctor prescribed Metformin and Amlodipine.", MEDICATION=["Metformin", "Amlodipine"]),
    make_example("The patient is also receiving chemotherapy.", TREATMENT=["chemotherapy"]),
    make_example("Asthma patients often carry Salbutamol.", DISEASE=["Asthma"], MEDICATION=["Salbutamol"]),
    make_example("Insulin and Metformin are used against diabetes.", MEDICATION=["Insulin", "Metformin"], DISEASE=["diabetes"]),
    make_example("Radiotherapy and surgery are options for cancer.", TREATMENT=["Radiotherapy", "surgery"], DISEASE=["cancer"]),
    make_example("He recovered from pneumonia after taking Amoxicillin.", DISEASE=["pneumonia"], MEDICATION=["Amoxicillin"]),
    make_example("Physiotherapy reduced her migraine.", TREATMENT=["Physiotherapy"], DISEASE=["migraine"]),
]

print("Training sentences:", len(TRAIN_DATA))
print("Testing sentences :", len(TEST_DATA))
print("\nFirst training example:")
print(TRAIN_DATA[0])

In [ ]:
random.seed(7)

custom_nlp = spacy.blank("en")
ner = custom_nlp.add_pipe("ner")

# Add custom entity labels
for label in ["DISEASE", "MEDICATION", "TREATMENT"]:
    ner.add_label(label)
print("Blank model created, custom labels added:", ner.labels)

train_examples = [Example.from_dict(custom_nlp.make_doc(t), a) for t, a in TRAIN_DATA]
test_examples  = [Example.from_dict(custom_nlp.make_doc(t), a) for t, a in TEST_DATA]

optimizer = custom_nlp.initialize(lambda: train_examples)
loss_history = []

for iteration in range(40):
    random.shuffle(train_examples)
    losses = {}
    for batch in spacy.util.minibatch(train_examples, size=4):
        custom_nlp.update(batch, drop=0.3, sgd=optimizer, losses=losses)
    loss_history.append(losses["ner"])
    if iteration % 5 == 0:
        print(f"Iteration {iteration:>2}, Loss: {losses['ner']:.4f}")

plt.figure(figsize=(7, 4))
plt.plot(loss_history)
plt.xlabel("Iteration")
plt.ylabel("NER loss")
plt.title("Training loss of the custom NER model")
plt.show()

In [ ]:
test_text = """
The patient has diabetes and hypertension.
The doctor prescribed Metformin and Amlodipine.
The patient is also receiving chemotherapy.
"""

doc_custom = custom_nlp(test_text)

for ent in doc_custom.ents:
    print(f"{ent.text:<15} → {ent.label_}")

displacy.render(doc_custom, style="ent", jupyter=True)

## 3. Quick alternative: add rules to the pre-trained model (EntityRuler)

Instead of training, word lists can be added to the pre-trained pipeline. This needs no training data but only
finds the words that are in the list.

In [ ]:
ruler_nlp = spacy.load("en_core_web_sm")
ruler = ruler_nlp.add_pipe("entity_ruler", before="ner")

diseases    = ["diabetes", "hypertension", "asthma", "cancer", "fever", "migraine", "pneumonia", "arthritis", "malaria", "stroke"]
medications = ["metformin", "amlodipine", "insulin", "salbutamol", "paracetamol", "aspirin", "ibuprofen", "amoxicillin"]
treatments  = ["chemotherapy", "radiotherapy", "physiotherapy", "surgery", "dialysis", "vaccination"]

patterns = (
    [{"label": "DISEASE",    "pattern": [{"LOWER": w}]} for w in diseases] +
    [{"label": "MEDICATION", "pattern": [{"LOWER": w}]} for w in medications] +
    [{"label": "TREATMENT",  "pattern": [{"LOWER": w}]} for w in treatments]
)
ruler.add_patterns(patterns)

doc_rules = ruler_nlp(test_text)
for ent in doc_rules.ents:
    print(f"{ent.text:<15} → {ent.label_}")

## 4. Pre-trained model on a longer news paragraph

In [ ]:
news = """
Prime Minister Narendra Modi met President Vladimir Putin in Moscow on Tuesday to discuss energy trade.
Officials from the Reserve Bank of India said imports of discounted oil rose by 12 per cent last year.
The talks, reported by Reuters, are expected to continue in New Delhi in October.
"""

news_doc = nlp(news)
for ent in news_doc.ents:
    print(f"{ent.text:<30} {ent.label_}")

displacy.render(news_doc, style="ent", jupyter=True)

---
# Testing the NER models

* **Test 1** - the pre-trained model against hand-labelled sentences (precision, recall, F1).
* **Test 2** - the custom model on held-out medical sentences it never saw during training.
* **Test 3** - custom model vs. rule-based EntityRuler.
* **Test 4** - generalisation: medical words the custom model has never seen.

*Precision* = of the entities found, how many are right. *Recall* = of the real entities, how many were found.
*F1* = the balance of both. An entity counts as correct only when its **text and label** both match.

In [ ]:
def prf(gold, predicted):
    """Entity-level precision, recall and F1 from two sets of (text, label)."""
    tp = len(gold & predicted)
    precision = tp / len(predicted) if predicted else 0.0
    recall = tp / len(gold) if gold else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return precision, recall, f1

def evaluate(model, labelled_sentences):
    """labelled_sentences: list of (text, [(entity text, label), ...])."""
    all_gold, all_pred, per_label, detail = set(), set(), {}, []
    for i, (sentence, gold_entities) in enumerate(labelled_sentences):
        gold = {(i, t, l) for t, l in gold_entities}
        pred = {(i, e.text, e.label_) for e in model(sentence).ents}
        all_gold |= gold
        all_pred |= pred
        detail.append({
            "Sentence": sentence[:60],
            "Missed": [(t, l) for _, t, l in sorted(gold - pred)],
            "Wrong / extra": [(t, l) for _, t, l in sorted(pred - gold)],
        })
        for _, t, l in gold | pred:
            per_label.setdefault(l, [set(), set()])
        for g in gold: per_label[g[2]][0].add(g)
        for p in pred: per_label[p[2]][1].add(p)
    rows = []
    for label, (g, p) in sorted(per_label.items()):
        pr, rc, f = prf(g, p)
        rows.append({"Label": label, "Gold": len(g), "Predicted": len(p),
                     "Precision": round(pr, 2), "Recall": round(rc, 2), "F1": round(f, 2)})
    overall = prf(all_gold, all_pred)
    return overall, pd.DataFrame(rows), pd.DataFrame(detail)

## Test 1: pre-trained model on hand-labelled sentences

In [ ]:
GOLD_NEWS = [
    ("Sundar Pichai said Google will open a new office in Hyderabad in 2027.",
     [("Sundar Pichai", "PERSON"), ("Google", "ORG"), ("Hyderabad", "GPE"), ("2027", "DATE")]),
    ("Tesla delivered 400,000 cars in the first quarter, Elon Musk told Reuters.",
     [("Tesla", "ORG"), ("400,000", "CARDINAL"), ("the first quarter", "DATE"), ("Elon Musk", "PERSON"), ("Reuters", "ORG")]),
    ("Oxford University and Microsoft signed a $10 million agreement in London on Monday.",
     [("Oxford University", "ORG"), ("Microsoft", "ORG"), ("$10 million", "MONEY"), ("London", "GPE"), ("Monday", "DATE")]),
    ("Prime Minister Narendra Modi visited Japan last week.",
     [("Narendra Modi", "PERSON"), ("Japan", "GPE"), ("last week", "DATE")]),
    ("Amazon plans to hire 5,000 people in Pune by December.",
     [("Amazon", "ORG"), ("5,000", "CARDINAL"), ("Pune", "GPE"), ("December", "DATE")]),
    ("Angela Merkel and Emmanuel Macron met in Paris on Friday.",
     [("Angela Merkel", "PERSON"), ("Emmanuel Macron", "PERSON"), ("Paris", "GPE"), ("Friday", "DATE")]),
]

overall, per_label_df, detail_df = evaluate(nlp, GOLD_NEWS)
print(f"Pre-trained model  ->  Precision {overall[0]:.2f}   Recall {overall[1]:.2f}   F1 {overall[2]:.2f}")
display(per_label_df)

print("\nMistakes (missed = real entity not found; wrong/extra = predicted but incorrect):")
display(detail_df[(detail_df["Missed"].str.len() > 0) | (detail_df["Wrong / extra"].str.len() > 0)])
pretrained_f1 = overall[2]

## Test 2: custom model on held-out medical sentences

In [ ]:
scores = custom_nlp.evaluate(test_examples)
print("Custom model (spaCy Scorer)  ->  Precision {:.2f}   Recall {:.2f}   F1 {:.2f}".format(
    scores["ents_p"], scores["ents_r"], scores["ents_f"]))

per_type = pd.DataFrame(scores["ents_per_type"]).T.round(2)
per_type.columns = ["Precision", "Recall", "F1"]
display(per_type)

rows = []
for sentence, ann in TEST_DATA:
    expected = [(sentence[s:e], l) for s, e, l in ann["entities"]]
    found = [(ent.text, ent.label_) for ent in custom_nlp(sentence).ents]
    rows.append({"Sentence": sentence, "Expected": expected, "Found": found,
                 "Correct": "yes" if set(expected) == set(found) else "NO"})
results_df = pd.DataFrame(rows)
display(results_df)
print(f"Sentences fully correct: {(results_df['Correct'] == 'yes').sum()} / {len(results_df)}")
custom_f1 = scores["ents_f"]

## Test 3: custom model vs. rule-based EntityRuler

In [ ]:
medical_gold = [(s, [(s[a:b], l) for a, b, l in ann["entities"]]) for s, ann in TEST_DATA]

(_, _, f_custom) = evaluate(custom_nlp, medical_gold)[0]
(p_rule, r_rule, f_rule) = evaluate(ruler_nlp, medical_gold)[0]
(p_cust, r_cust, _) = evaluate(custom_nlp, medical_gold)[0]

compare = pd.DataFrame({
    "Approach": ["Trained custom NER", "Rule-based EntityRuler", "Pre-trained model only (no medical labels)"],
    "Precision": [round(p_cust, 2), round(p_rule, 2), round(evaluate(nlp, medical_gold)[0][0], 2)],
    "Recall":    [round(r_cust, 2), round(r_rule, 2), round(evaluate(nlp, medical_gold)[0][1], 2)],
    "F1":        [round(f_custom, 2), round(f_rule, 2), round(evaluate(nlp, medical_gold)[0][2], 2)],
})
display(compare)
print("Note: the EntityRuler word list was written with these test words in mind, so its score is optimistic;")
print("it would miss any disease or drug that is not in the list (see Test 4 for the trained model on new words).")

compare.set_index("Approach")[["Precision", "Recall", "F1"]].plot(kind="bar", figsize=(9, 4), ylim=(0, 1.05))
plt.title("Medical entities - three approaches on the same test sentences")
plt.xticks(rotation=10)
plt.show()

## Test 4: generalisation to words the model has never seen

In [ ]:
UNSEEN = [
    make_example("Lupus is treated with Hydroxychloroquine.", DISEASE=["Lupus"], MEDICATION=["Hydroxychloroquine"]),
    make_example("The patient with tuberculosis received Rifampicin.", DISEASE=["tuberculosis"], MEDICATION=["Rifampicin"]),
    make_example("Acupuncture may relieve chronic back pain.", TREATMENT=["Acupuncture"], DISEASE=["back pain"]),
]
unseen_gold = [(s, [(s[a:b], l) for a, b, l in ann["entities"]]) for s, ann in UNSEEN]

(p, r, f), unseen_labels, unseen_detail = evaluate(custom_nlp, unseen_gold)
print(f"Custom model on UNSEEN medical words  ->  Precision {p:.2f}   Recall {r:.2f}   F1 {f:.2f}")
display(unseen_detail)
print("A model trained on only 24 sentences remembers the words it has seen; a few hundred more examples")
print("are needed before it recognises new drug and disease names reliably.")

## Test 5: try your own text

In [ ]:
def extract_entities(text, model=None):
    model = model or nlp
    d = model(text)
    if not d.ents:
        print("(no entities found)")
    for ent in d.ents:
        print(f"{ent.text:<28} {ent.label_:<11} {spacy.explain(ent.label_) or ''}")

print("--- pre-trained model ---")
extract_entities("Infosys will open an office in Pune next March, said Nandan Nilekani.")

print("\n--- custom medical model ---")
extract_entities("The doctor recommended Insulin and dialysis for the patient with diabetes.", custom_nlp)

## Conclusion

Entities were extracted from news text with a pre-trained spaCy model and displayed with displaCy. A custom
model was then trained for DISEASE, MEDICATION and TREATMENT entities. The testing section measured precision,
recall and F1 for the pre-trained model (hand-labelled news sentences), the custom model (held-out medical
sentences) and a rule-based alternative, and showed how the custom model behaves on words it has never seen.